# Image Captioning: Inference and Analysis
This notebook demonstrates the trained model on a held-out test split, showcasing successes and failures.

In [ ]:
import os
import sys
import random
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Ensure src is in the path
sys.path.append(os.path.abspath('..'))

from src.vocabulary import Vocabulary
from src.dataset import get_transform, split_by_image, load_captions_df

# Constants and Paths
DATA_DIR = '../data/caption_data'
IMAGES_DIR = os.path.join(DATA_DIR, 'Images')
CAPTIONS_FILE = os.path.join(DATA_DIR, 'captions.txt')
CHECKPOINT_PATH = '../checkpoints/best_model.pth'
VOCAB_PATH = '../checkpoints/vocab.pkl'


## 1. Setup: Load Vocab and Model

In [ ]:
# Load Vocabulary
try:
    vocab = Vocabulary.load(VOCAB_PATH)
    print(f"Vocabulary loaded successfully. Size: {len(vocab)}")
except FileNotFoundError:
    print("Vocabulary file not found. Ensure training has been completed.")
    vocab = None

# Load Model
try:
    # TODO: depends on Person B's utils.load_checkpoint()
    from src import utils
    model = utils.load_checkpoint(CHECKPOINT_PATH)
    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model (expected if checkpoint not created yet): {e}")
    model = None

## 2. Generate Caption Function

In [ ]:
def generate_caption(image_path: str, model: any) -> str:
    """
    Takes a path to an image and returns a generated caption string.
    """
    if model is None or vocab is None:
        return "Model or vocab not loaded."
        
    # Reuse the test transform from dataset.py
    transform = get_transform(train=False)
    
    # Load and transform the image
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)  # Add batch dim -> [1, 3, 224, 224]
    
    # Push to device
    device = next(model.parameters()).device if model else torch.device('cpu')
    image_tensor = image_tensor.to(device)
    
    # Generate sequence (expected from Person B's model.py)
    with torch.no_grad():
        output = model.generate(image_tensor, vocab, max_len=20)
        
    # Handle output types cleanly
    if isinstance(output, str):
        return output
    elif isinstance(output, list) or isinstance(output, torch.Tensor):
        return vocab.denumericalize(output)
    else:
        return str(output)

## 3. Demonstration on Held-out Test Split

In [ ]:
# Load data and isolate test split to prevent data leakage
if os.path.exists(CAPTIONS_FILE):
    df = load_captions_df(CAPTIONS_FILE)
    _, _, test_df = split_by_image(df, val_frac=0.1, test_frac=0.1, seed=42)
    test_images = test_df['image'].unique()
    
    # Pick 8 random test images
    random.seed(42)
    sample_images = random.sample(list(test_images), min(8, len(test_images)))
    
    # Plotting
    fig, axes = plt.subplots(4, 2, figsize=(15, 20))
    axes = axes.flatten()
    
    for i, image_name in enumerate(sample_images):
        img_path = os.path.join(IMAGES_DIR, image_name)
        
        # Generate caption
        caption = generate_caption(img_path, model)
        
        # Display image
        img = Image.open(img_path).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f"Generated: {caption}", wrap=True, fontsize=12)
        axes[i].axis('off')
        
        # Print ground truths for manual inspection
        gts = test_df[test_df['image'] == image_name]['caption'].tolist()
        print(f"\n--- Image: {image_name} ---")
        print(f"Generated: {caption}")
        print("Ground Truths:")
        for gt in gts:
            print(f"  - {gt}")
            
    plt.tight_layout()
    plt.show()
else:
    print(f"Dataset not found at {CAPTIONS_FILE}. Please verify data directory.")

## 4. Analysis — Successes

*(Hypothetical analysis based on expected model performance on Flickr8k)*

**1. Common Scene/Object:** 
- *Image:* A dog catching a frisbee in the park.
- *Why it succeeded:* The Flickr8k dataset contains numerous examples of dogs, grass, and frisbees. The model likely formed strong associations between these visual features and their corresponding text tokens due to high representation in the training split. The simple, uncluttered composition also made it easy for the CNN to extract clear features.

**2. Color/Attribute Recognition:**
- *Image:* A young child in a red jacket sliding down a slide.
- *Why it succeeded:* The model effectively localized specific attributes (the red jacket) and associated them with the correct noun. This indicates that the attention mechanism successfully learned to isolate specific regions when predicting sequential adjectives and nouns.

**3. Multiple Entity Interaction:**
- *Image:* Two men riding bicycles on a dirt path.
- *Why it succeeded:* The model correctly recognized multiple entities and their actions. The strong co-occurrence of "people", "riding", and "bikes" in the training corpus helps the LSTM component formulate grammatically accurate sequences when triggered by the right visual cues.

## 5. Analysis — Failures

*(Hypothetical analysis of typical failure modes for encoder-decoder models)*

**1. Repetition Loop (Exposure Bias):**
- *Image:* A man in a blue shirt standing near a building.
- *Why it failed:* This is a common failure mode when using greedy decoding with LSTMs. The model got stuck in a repetitive loop predicting "in a blue shirt" and lacked the broader context or sequence-level planning to realize it had already generated that phrase.

**2. Fine-grained Category Confusion:**
- *Image:* A brown horse jumping over a fence.
- *Why it failed:* The model confused a horse with a dog, likely due to similar poses (jumping) and a visually ambiguous angle. The dataset has significantly more dogs than horses, biasing the model toward the majority class when uncertain about the specific animal type.

**3. Hallucinating Context:**
- *Image:* A person in a heavy coat standing on a rocky cliff.
- *Why it failed:* The model hallucinated a "snowy mountain" context. This could happen if the lighting or white clouds in the background triggered visual features often associated with snow in the training set. The model over-relied on typical visual co-occurrences (heavy coats often appear with snow) rather than strict visual evidence.

## 6. Optional: Attention Heatmaps

Below we visualize where the model focuses its attention while generating each word.

In [ ]:
# TODO: depends on Person B's model exposing attention weights during generate().
# Example usage (commented out until model.generate_with_attention is implemented):

'''
from src.utils import denormalize_image
import math

def plot_attention(image_path, result_caption, attention_plot):
    temp_image = np.array(Image.open(image_path))
    
    fig = plt.figure(figsize=(10, 10))
    len_result = len(result_caption)
    
    for l in range(len_result):
        temp_att = np.resize(attention_plot[l], (8, 8))
        ax = fig.add_subplot(math.ceil(len_result/2), 2, l+1)
        ax.set_title(result_caption[l])
        img = ax.imshow(temp_image)
        ax.imshow(temp_att, cmap='gray', alpha=0.6, extent=img.get_extent())
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
'''